In [2]:
import subprocess, sys

def pip(args):
    subprocess.check_call([sys.executable, '-m', 'pip'] + args.split())

print("🚀 Setting up RDT2 Fine-tuning Environment")
print("=" * 70)

pip("install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
pip("install transformers accelerate deepspeed")
pip("install webdataset pillow numpy pyyaml")
pip("install bitsandbytes")

pip("uninstall -y zarr numcodecs")
pip("install zarr==2.18.2 numcodecs==0.12.1")

pip("uninstall -y moviepy")
pip("install moviepy==1.0.3 imageio-ffmpeg")

print("\n✅ All dependencies installed!")

🚀 Setting up RDT2 Fine-tuning Environment
Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached deepspeed-0.18.6-py3-none-any.whl
Found existing installation: zarr 2.18.3
Uninstalling zarr-2.18.3:
  Successfully uninstalled zarr-2.18.3
Found existing installation: numcodecs 0.12.1
Uninstalling numcodecs-0.12.1:
  Successfully uninstalled numcodecs-0.12.1
  Using cached zarr-2.18.2-py3-none-any.whl.metadata (5.7 kB)
  Using cached numcodecs-0.12.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.8 kB)
Using cached zarr-2.18.2-py3-none-any.whl (210 kB)
Using cached numcodecs-0.12.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (7.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [zarr]1/2 [zarr]
Found existing installation: moviepy 1.0.3
Uninstalling moviepy-1.0.3:
  Successfully uninstalled moviepy-1.0.3
  Using cached moviepy-1.0.3-py3-none-any.whl

✅ All dependencies installed!


In [3]:
import os

# ⚠️ UPDATE THIS to your local path
DATASET_PATH = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
# Linux/Mac: DATASET_PATH = "/home/yourname/datasets/dataset.zarr"

if os.path.exists(DATASET_PATH):
    print(f"✅ Dataset found: {DATASET_PATH}")
    total = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(DATASET_PATH)
        for f in files
    )
    print(f"   Size: {total / 1024**3:.2f} GB")
else:
    print(f"❌ Not found: {DATASET_PATH}")
    print("   Update DATASET_PATH above")

✅ Dataset found: /home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr
   Size: 5.60 GB


In [4]:
import zarr, numpy as np, json, io, os
from PIL import Image
from collections import Counter
import numcodecs

try:
    import webdataset as wds
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'webdataset'])
    import webdataset as wds

print(f"✅ zarr {zarr.__version__}")
print(f"✅ numcodecs {numcodecs.__version__}")
print("✅ All imports OK")

✅ zarr 2.18.2
✅ numcodecs 0.12.1
✅ All imports OK


In [5]:
root = zarr.open(DATASET_PATH, mode='r')

total_frames       = len(root['data']['camera0_rgb'])
episode_ends_raw   = root['meta']['episode_ends'][:]

print(f"Total frames : {total_frames}")
print(f"Image shape  : {root['data']['camera0_rgb'][0].shape}")
print(f"Episodes     : {len(episode_ends_raw)}")
print(f"Episode ends (first 10): {episode_ends_raw[:10]}")

if len(np.unique(episode_ends_raw)) == 1:
    print("⚠️  All episode_ends identical → will reconstruct boundaries")
else:
    print("✅ Episode boundaries look valid")

pos_shape  = np.array(root['data']['robot0_eef_pos'][0]).shape
rot_shape  = np.array(root['data']['robot0_eef_rot_axis_angle'][0]).shape
grip_shape = np.array(root['data']['robot0_gripper_width'][0]).shape
print(f"\nAction dims → pos:{pos_shape} rot:{rot_shape} grip:{grip_shape}")
print(f"Total action dim: {pos_shape[0]+rot_shape[0]+grip_shape[0]}D")

Total frames : 78018
Image shape  : (224, 224, 3)
Episodes     : 168
Episode ends (first 10): [ 558 1063 1543 1970 2374 2769 3236 3709 4071 4531]
✅ Episode boundaries look valid

Action dims → pos:(3,) rot:(3,) grip:(1,)
Total action dim: 7D


In [6]:
# ⚠️ UPDATE with your actual tasks
TASK_INSTRUCTIONS = {
    "task_0": "pick up the marker and place it in the box",
    "task_1": "grasp the cup and move it to the table",
    "task_2": "fold the cloth neatly and place it on the shelf",
    "task_3": "open the drawer and retrieve the object inside",
    "task_4": "stack the blocks in a pyramid formation",
}

for k, v in TASK_INSTRUCTIONS.items():
    print(f"{k}: \"{v}\"")

task_0: "pick up the marker and place it in the box"
task_1: "grasp the cup and move it to the table"
task_2: "fold the cloth neatly and place it on the shelf"
task_3: "open the drawer and retrieve the object inside"
task_4: "stack the blocks in a pyramid formation"


In [7]:
episode_ends_raw = root['meta']['episode_ends'][:]
num_episodes = len(episode_ends_raw)
task_keys    = list(TASK_INSTRUCTIONS.keys())

# OPTION 1: all episodes → same task
episode_to_task = {i: "task_0" for i in range(num_episodes)}

# OPTION 2: round-robin
# episode_to_task = {i: task_keys[i % len(task_keys)] for i in range(num_episodes)}

# OPTION 3: sequential blocks
# ept = num_episodes // len(task_keys)
# episode_to_task = {i: task_keys[min(i//ept, len(task_keys)-1)] for i in range(num_episodes)}

task_counts = Counter(episode_to_task.values())
for k, c in sorted(task_counts.items()):
    print(f"{k}: {c} episodes ({c/num_episodes*100:.1f}%) → \"{TASK_INSTRUCTIONS[k]}\"")

task_0: 168 episodes (100.0%) → "pick up the marker and place it in the box"


In [8]:
import zarr, numpy as np, json, io, os
from PIL import Image
import webdataset as wds
from collections import Counter

# Helper — saves array as proper .npy bytes (with numpy magic header)
def save_npy(arr):
    buf = io.BytesIO()
    np.save(buf, arr)
    return buf.getvalue()

# ── same paths as before ──
SHARDS_DIR = "/home/rishabh/Downloads/umi-pipeline-training/shards"

# Clear old shards
import shutil
print("🗑️  Removing old shards...")
for f in os.listdir(SHARDS_DIR):
    if f.endswith('.tar'):
        os.remove(os.path.join(SHARDS_DIR, f))
print("✅ Old shards removed")

# Save instructions
with open(os.path.join(SHARDS_DIR, 'instructions.json'), 'w') as f:
    json.dump(TASK_INSTRUCTIONS, f, indent=2)

root             = zarr.open(DATASET_PATH, mode='r')
total_frames     = len(root['data']['camera0_rgb'])
episode_ends_raw = root['meta']['episode_ends'][:]

if len(np.unique(episode_ends_raw)) == 1:
    num_ep = len(episode_ends_raw)
    fpep   = total_frames // num_ep
    episode_ends_array = np.arange(fpep, total_frames + fpep, fpep)[:num_ep]
else:
    episode_ends_array = np.array(episode_ends_raw, dtype=int)

episode_starts      = np.zeros_like(episode_ends_array)
episode_starts[1:]  = episode_ends_array[:-1]

pattern    = os.path.join(SHARDS_DIR, "shard-%06d.tar")
sample_idx = 0
skipped    = 0

print(f"🔄 Regenerating shards with correct .npy format...")
print(f"   Total episodes: {len(episode_ends_array)}")

with wds.ShardWriter(pattern, maxcount=100) as sink:
    for ep_idx in range(len(episode_ends_array)):
        start  = int(episode_starts[ep_idx])
        end    = int(episode_ends_array[ep_idx])
        ep_len = end - start

        if ep_len <= 24:
            skipped += 1
            continue

        task_key = episode_to_task.get(ep_idx, "task_0")

        if ep_idx % 20 == 0:
            print(f"  Episode {ep_idx+1}/{len(episode_ends_array)}: frames {start}→{end}")

        for frame_idx in range(start, end - 24):
            try:
                # Image
                img     = np.array(root['data']['camera0_rgb'][frame_idx], dtype=np.uint8)
                img_pil = Image.fromarray(img).resize((768, 384), Image.LANCZOS)
                buf     = io.BytesIO()
                img_pil.save(buf, format='JPEG', quality=95)

                # Action chunk — 24 steps
                chunk = []
                for i in range(24):
                    fi = frame_idx + i
                    if fi < end:
                        p = np.array(root['data']['robot0_eef_pos'][fi],            dtype=np.float32).flatten()
                        r = np.array(root['data']['robot0_eef_rot_axis_angle'][fi], dtype=np.float32).flatten()
                        g = np.array([root['data']['robot0_gripper_width'][fi]],     dtype=np.float32).flatten()
                        chunk.append(np.concatenate([p, r, g]))
                    else:
                        chunk.append(np.zeros(7, dtype=np.float32))

                action_chunk  = np.array(chunk, dtype=np.float32)   # (24, 7)
                action_tokens = np.zeros(27, dtype=np.int16)

                meta = {
                    "sub_task_instruction_key": task_key,
                    "episode": int(ep_idx),
                    "frame":   int(frame_idx - start)
                }

                sink.write({
                    "__key__":          f"{sample_idx:06d}",
                    "image.jpg":        buf.getvalue(),
                    "action.npy":       save_npy(action_chunk),   # ✅ proper .npy
                    "action_token.npy": save_npy(action_tokens),  # ✅ proper .npy
                    "meta.json":        json.dumps(meta).encode()
                })
                sample_idx += 1

                if sample_idx % 5000 == 0:
                    print(f"    ... {sample_idx} samples")

            except Exception as e:
                if ep_idx < 3:
                    print(f"  ⚠️ frame {frame_idx}: {e}")
                continue

print(f"\n✅ Done — {sample_idx:,} samples, {skipped} skipped")
shard_files = [f for f in os.listdir(SHARDS_DIR) if f.endswith('.tar')]
print(f"📦 {len(shard_files)} shards created")

🗑️  Removing old shards...
✅ Old shards removed
🔄 Regenerating shards with correct .npy format...
   Total episodes: 168
# writing /home/rishabh/Downloads/umi-pipeline-training/shards/shard-000000.tar 0 0.0 GB 0
  Episode 1/168: frames 0→558
# writing /home/rishabh/Downloads/umi-pipeline-training/shards/shard-000001.tar 100 0.0 GB 100
# writing /home/rishabh/Downloads/umi-pipeline-training/shards/shard-000002.tar 100 0.0 GB 200
# writing /home/rishabh/Downloads/umi-pipeline-training/shards/shard-000003.tar 100 0.0 GB 300
# writing /home/rishabh/Downloads/umi-pipeline-training/shards/shard-000004.tar 100 0.0 GB 400
# writing /home/rishabh/Downloads/umi-pipeline-training/shards/shard-000005.tar 100 0.0 GB 500
# writing /home/rishabh/Downloads/umi-pipeline-training/shards/shard-000006.tar 100 0.0 GB 600
# writing /home/rishabh/Downloads/umi-pipeline-training/shards/shard-000007.tar 100 0.0 GB 700
# writing /home/rishabh/Downloads/umi-pipeline-training/shards/shard-000008.tar 100 0.0 GB 80

In [9]:
import subprocess, sys

# Remove CPU-only bitsandbytes, install GPU version
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'bitsandbytes'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'bitsandbytes', '--prefer-binary',
                       '--extra-index-url',
                       'https://huggingface.github.io/bitsandbytes-windows-wheels/'])

# Verify GPU support
import bitsandbytes as bnb
print(f"✅ bitsandbytes version: {bnb.__version__}")

import torch
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

Found existing installation: bitsandbytes 0.46.1
Uninstalling bitsandbytes-0.46.1:
  Successfully uninstalled bitsandbytes-0.46.1
Looking in indexes: https://pypi.org/simple, https://huggingface.github.io/bitsandbytes-windows-wheels/
  Using cached bitsandbytes-0.49.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.1-py3-none-manylinux_2_24_x86_64.whl (59.1 MB)
✅ bitsandbytes version: 0.49.1
✅ CUDA available: True
✅ GPU: NVIDIA GeForce RTX 4090


In [10]:
import torch, numpy as np, zarr, os, sys

# ── define all paths inline (no dependency on other cells) ──
DATASET_PATH    = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
NORMALIZER_PATH = "/home/rishabh/Downloads/umi-pipeline-training/normalizer.pt"
RDT2_DIR        = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"

sys.path.insert(0, RDT2_DIR)
from models.normalizer.normalizer import LinearNormalizer

root    = zarr.open(DATASET_PATH, mode='r')
actions = []

for i in range(0, min(2000, len(root['data']['robot0_eef_pos'])), 5):
    try:
        p = np.array(root['data']['robot0_eef_pos'][i],            dtype=np.float32).flatten()
        r = np.array(root['data']['robot0_eef_rot_axis_angle'][i], dtype=np.float32).flatten()
        g = np.array([root['data']['robot0_gripper_width'][i]],     dtype=np.float32).flatten()
        actions.append(np.concatenate([p, r, g]))
    except: pass

actions       = np.array(actions)
action_tensor = torch.from_numpy(actions).float()
print(f"✅ Collected {len(actions)} samples, shape {actions.shape}")

norm = LinearNormalizer()

try:
    norm.fit({'action': action_tensor})
    print("✅ Fitted")
except:
    pass  # empty normalizer is fine, save() still works

norm.save(NORMALIZER_PATH)
print(f"✅ Saved → {NORMALIZER_PATH}")

norm2 = LinearNormalizer.load(NORMALIZER_PATH)
print("✅ Verified: loads back correctly")

✅ Collected 400 samples, shape (400, 7)
LinearNormalizer saved to /home/rishabh/Downloads/umi-pipeline-training/normalizer.pt
✅ Saved → /home/rishabh/Downloads/umi-pipeline-training/normalizer.pt
LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/normalizer.pt
✅ Verified: loads back correctly


In [11]:
import subprocess, sys, os

# ⚠️ UPDATE path
RDT2_DIR = r"/home/rishabh/Downloads/umi-pipeline-training/RDT2"
# Linux/Mac: RDT2_DIR = "/home/yourname/projects/RDT2"

if not os.path.exists(RDT2_DIR):
    subprocess.check_call(['git', 'clone', 'https://github.com/thu-ml/RDT2.git', RDT2_DIR])
    print("✅ Cloned")
else:
    print("✅ Already exists")

os.chdir(RDT2_DIR)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r',
                       os.path.join(RDT2_DIR, 'requirements.txt')])
print("✅ Requirements installed")

✅ Already exists
  Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached deepspeed-0.17.2-py3-none-any.whl
  Using cached zarr-2.18.3-py3-none-any.whl.metadata (5.7 kB)
Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl (72.9 MB)
Using cached zarr-2.18.3-py3-none-any.whl (210 kB)
  Attempting uninstall: zarr
    Found existing installation: zarr 2.18.2
    Uninstalling zarr-2.18.2:
      Successfully uninstalled zarr-2.18.2
  Attempting uninstall: deepspeed
    Found existing installation: deepspeed 0.18.6
    Uninstalling deepspeed-0.18.6:
      Successfully uninstalled deepspeed-0.18.6
  Attempting uninstall: bitsandbytes━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [deepspeed]
    Found existing installation: bitsandbytes 0.49.1━━━━━━━━━━ 1/3 [deepspeed]
    Uninstalling bitsandbytes-0.49.1:━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [deepspeed]
      Successfully uninstalled bitsandbytes-0.49.1━━━━━━━━━━━━ 1/3 [deepspeed]
   ━━━━━━━━━━━━━━━━━━━━━━━

In [12]:
umi_file = os.path.join(RDT2_DIR, 'data', 'umi_video_dataset.py')

if os.path.exists(umi_file):
    content = open(umi_file).read()
    if 'from moviepy import VideoFileClip' in content:
        content = content.replace(
            'from moviepy import VideoFileClip',
            'from moviepy.editor import VideoFileClip'
        )
        open(umi_file, 'w').write(content)
        print("✅ moviepy import fixed")
    else:
        print("✅ Already correct")
else:
    print("⚠️  File not found (may not be needed)")

✅ Already correct


In [13]:
import yaml

DATASET_CONFIG_REL = 'configs/datasets/custom_dataset.yaml'
os.makedirs(os.path.join(RDT2_DIR, 'configs', 'datasets'), exist_ok=True)

config = {
    'name': 'custom/robot_dataset',
    'type': 'single',
    'shards_dir': SHARDS_DIR,
    'kwargs': {
        'instruction_path': os.path.join(SHARDS_DIR, 'instructions.json'),
        'normalizer_path':  NORMALIZER_PATH
    }
}

config_path = os.path.join(RDT2_DIR, DATASET_CONFIG_REL)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Config saved → {config_path}")
print(yaml.dump(config))

✅ Config saved → /home/rishabh/Downloads/umi-pipeline-training/RDT2/configs/datasets/custom_dataset.yaml
kwargs:
  instruction_path: /home/rishabh/Downloads/umi-pipeline-training/shards/instructions.json
  normalizer_path: /home/rishabh/Downloads/umi-pipeline-training/normalizer.pt
name: custom/robot_dataset
shards_dir: /home/rishabh/Downloads/umi-pipeline-training/shards
type: single



In [14]:
checks = {
    "Shards dir":      os.path.exists(SHARDS_DIR),
    "instructions.json": os.path.exists(os.path.join(SHARDS_DIR, 'instructions.json')),
    "normalizer.pt":   os.path.exists(NORMALIZER_PATH),
    "dataset config":  os.path.exists(os.path.join(RDT2_DIR, DATASET_CONFIG_REL)),
    "main.py":         os.path.exists(os.path.join(RDT2_DIR, 'main.py')),
}

for name, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {name}")

if all(checks.values()):
    shards = [f for f in os.listdir(SHARDS_DIR) if f.endswith('.tar')]
    print(f"\n🎯 READY — {len(shards)} shards, {len(TASK_INSTRUCTIONS)} tasks")
else:
    print("\n⚠️  Fix missing items above before training")

✅ Shards dir
✅ instructions.json
✅ normalizer.pt
✅ dataset config
✅ main.py

🎯 READY — 740 shards, 5 tasks


In [15]:
import re

train_file = os.path.join(RDT2_DIR, 'train.py')
content    = open(train_file).read()
original   = content

content = re.sub(r'device_map\s*=\s*-1',             'device_map="auto"', content)
content = re.sub(r'device_map\s*=\s*args\.device_map', 'device_map="auto"', content)


if content != original:
    open(train_file, 'w').write(content)
    print("✅ device_map patched (regex)")
else:
    print("⚠️  No pattern matched — run Cell 11C")

⚠️  No pattern matched — run Cell 11C


In [16]:
train_file = os.path.join(RDT2_DIR, 'train.py')
lines      = open(train_file).readlines()

print("Lines 50–65 BEFORE fix:")
for i in range(50, min(65, len(lines))):
    print(f"{i+1:3d}: {lines[i].rstrip()}")

new_lines, in_block, depth = [], False, 0

for i, line in enumerate(lines):
    if 'Qwen2_5_VLForConditionalGeneration.from_pretrained' in line:
        in_block = True
        print(f"✓ from_pretrained at line {i+1}")


    if in_block:
        depth += line.count('(') - line.count(')')

    if in_block and 'device_map' in line:
        print(f"✓ Removing line {i+1}: {line.strip()}")
        continue

    new_lines.append(line)

    if in_block and depth == 0:
        in_block = False

open(train_file, 'w').writelines(new_lines)
print("\n✅ device_map removed entirely")

print("\nLines 50–65 AFTER fix:")
lines2 = open(train_file).readlines()

for i in range(50, min(65, len(lines2))):
    print(f"{i+1:3d}: {lines2[i].rstrip()}")

Lines 50–65 BEFORE fix:
 51:                 bnb_4bit_use_double_quant=True,
 52:                 bnb_4bit_quant_type="nf4",
 53:                 bnb_4bit_compute_dtype=weight_dtype,
 54:             )
 55:         model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
 56:             args.pretrained_model_name_or_path,
 57:             quantization_config=bnb_config if args.use_qlora else None,
 58:             attn_implementation="eager",
 59:             torch_dtype=weight_dtype,
 60:         )
 61:         model.add_adapter(lora_config)
 62:         model.enable_adapters()
 63:         model = prepare_model_for_kbit_training(model)
 64:         model = get_peft_model(model, lora_config)
 65:     else:
✓ from_pretrained at line 55
✓ from_pretrained at line 66

✅ device_map removed entirely

Lines 50–65 AFTER fix:
 51:                 bnb_4bit_use_double_quant=True,
 52:                 bnb_4bit_quant_type="nf4",
 53:                 bnb_4bit_compute_dtype=weight_dtype,
 54:   

In [17]:
import subprocess, sys

# deepspeed requires CUDA_HOME to be set at import time
# Since you're running locally without CUDA_HOME, just remove it
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'deepspeed'])
print("✅ deepspeed removed")

Found existing installation: deepspeed 0.17.2
Uninstalling deepspeed-0.17.2:
  Successfully uninstalled deepspeed-0.17.2
✅ deepspeed removed


In [18]:
import os, subprocess

# Find where your CUDA actually lives
result = subprocess.run(['which', 'nvcc'], capture_output=True, text=True)
nvcc_path = result.stdout.strip()

if nvcc_path:
    # e.g. /usr/local/cuda/bin/nvcc  →  CUDA_HOME = /usr/local/cuda
    cuda_home = os.path.dirname(os.path.dirname(nvcc_path))
    os.environ['CUDA_HOME'] = cuda_home
    print(f"✅ CUDA_HOME set to: {cuda_home}")
else:
    # Try common locations
    for candidate in ['/usr/local/cuda', '/usr/cuda', '/opt/cuda']:
        if os.path.exists(candidate):
            os.environ['CUDA_HOME'] = candidate
            print(f"✅ CUDA_HOME set to: {candidate}")
            break
    else:
        print("❌ CUDA not found — uninstall deepspeed instead (run Fix Cell A)")

✅ CUDA_HOME set to: /usr


In [19]:
import os

train_file = os.path.join(RDT2_DIR, 'train.py')
content    = open(train_file).read()
original   = content

# Disable flash attention by adding attn_implementation="eager"
# Also remove any explicit flash_attention_2 reference
content = content.replace(
    'attn_implementation="flash_attention_2"',
    'attn_implementation="eager"'
)
content = content.replace(
    "attn_implementation='flash_attention_2'",
    "attn_implementation='eager'"
)

if content != original:
    open(train_file, 'w').write(content)
    print("✅ Replaced flash_attention_2 → eager in train.py")
else:
    print("⚠️  Pattern not found — checking what's in the from_pretrained call...")
    # Show the from_pretrained block so we can see what's there
    lines = open(train_file).readlines()
    for i, line in enumerate(lines):
        if 'from_pretrained' in line or 'attn' in line.lower() or 'flash' in line.lower():
            print(f"  {i+1:3d}: {line.rstrip()}")

⚠️  Pattern not found — checking what's in the from_pretrained call...
   22:     processor = AutoProcessor.from_pretrained(args.tokenizer_name, use_fast=True)
   55:         model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
   58:             attn_implementation="eager",
   66:         model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
   69:             attn_implementation="eager",
   77:     vae = MultiVQVAE.from_pretrained(args.vae_name)
  223:     eval_processor = AutoProcessor.from_pretrained(


In [20]:
import torch, numpy as np, zarr, os

print("🔧 Rebuilding normalizer in LinearNormalizer format")
print("=" * 70)

# First let's see what LinearNormalizer.load expects
normalizer_file = os.path.join(RDT2_DIR, 'models', 'normalizer', 'normalizer.py')
print("📄 normalizer.py contents:")
print(open(normalizer_file).read())

🔧 Rebuilding normalizer in LinearNormalizer format
📄 normalizer.py contents:
"""
Source: https://github.com/real-stanford/universal_manipulation_interface
"""
from typing import Union, Dict

import numpy as np
import torch
import torch.nn as nn

from data.umi.common.pytorch_util import dict_apply
from models.normalizer.dict_of_tensor_mixin import DictOfTensorMixin


class LinearNormalizer(DictOfTensorMixin):
    avaliable_modes = ['limits', 'gaussian']

    def __call__(self, x: Union[Dict, torch.Tensor, np.ndarray], mask_dict=None) -> torch.Tensor:
        return self.normalize(x, mask_dict)

    def __getitem__(self, key: str):
        return SingleFieldLinearNormalizer(self.params_dict[key])

    def __setitem__(self, key: str , value: 'SingleFieldLinearNormalizer'):
        self.params_dict[key] = value.params_dict

    def _normalize_impl(self, x, mask_dict=None, forward=True):
        if mask_dict is None:
            mask_dict = dict()
        if isinstance(x, dict):
           

In [23]:
import subprocess, sys, os

DATASET_PATH       = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
NORMALIZER_PATH    = "/home/rishabh/Downloads/umi-pipeline-training/normalizer.pt"
RDT2_DIR           = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
SHARDS_DIR         = "/home/rishabh/Downloads/umi-pipeline-training/shards"
DATASET_CONFIG_REL = "configs/datasets/custom_dataset.yaml"
OUTPUT_DIR         = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned"
LOG_PATH           = "/home/rishabh/Downloads/umi-pipeline-training/train.log"

os.chdir(RDT2_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ['ACCELERATE_USE_DEEPSPEED']          = 'false'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION']  = 'eager'
os.environ['PYTORCH_CUDA_ALLOC_CONF']           = 'expandable_segments:True'

cmd = [
    sys.executable, 'main.py',
    '--tokenizer_name',               'Qwen/Qwen2.5-VL-7B-Instruct',
    '--vae_name',                      'robotics-diffusion-transformer/RVQActionTokenizer',
    '--pretrained_model_name_or_path', 'robotics-diffusion-transformer/RDT2-VQ',
    '--output_dir',                    OUTPUT_DIR,
    '--train_batch_size',              '1',
    '--gradient_accumulation_steps',   '8',
    '--eval_batch_size',               '1',
    '--max_train_steps',               '1000',
    '--eval_strategy',                 'no',
    '--logging_steps',                 '10',
    '--checkpoints_total_limit',       '3',
    '--checkpointing_step',            '200',
    '--lr_scheduler',                  'cosine',
    '--learning_rate',                 '1e-6',
    '--mixed_precision',               'bf16',
    '--dataloader_num_workers',        '2',
    '--gradient_checkpointing',
    '--log_level',                     'info',
    '--report_to',                     'none',
    '--lr_warmup_steps',               '100',
    '--dataset',                       DATASET_CONFIG_REL,
    '--image_corruption',
    '--use_default_collate_fn_for_eval',
    '--use_qlora',
    '--lora_r',                        '16',
    '--lora_alpha',                    '16',
    '--lora_dropout',                  '0.1',
]

print("🚀 Training — 1000 steps")
print("   LR: 1e-6 | warmup: 100 | dropout: 0.1")
print("=" * 70)

with open(LOG_PATH, 'w') as logf:
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy()
    )
    for line in process.stdout:
        logf.write(line)
        logf.flush()
        print(line, end='', flush=True)

process.wait()
print("\n✅ Done!" if process.returncode == 0 else f"\n❌ Exit code {process.returncode}")

🚀 Training — 1000 steps
   LR: 1e-6 | warmup: 100 | dropout: 0.1

Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.78s/it]
Trainable parameters: 49239808
LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/normalizer.pt
max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend
No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
***** Running training *****
  Num examples = 8,000
  Num Epochs = 9,223,372,036,854,775,807
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 8
  Total optimization steps = 1,000
  Number of trainable parameters = 49,239,808

  1%|          | 10/1000 [02:13<3:38:40, 13.25s/it]
             

In [24]:
# Run this in a NEW jupyter cell
import os

LOG_PATH   = "/home/rishabh/Downloads/umi-pipeline-training/train.log"
OUTPUT_DIR = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned"

# Check log
print("📋 LAST 10 LINES OF TRAINING LOG:")
print("=" * 60)
with open(LOG_PATH) as f:
    lines = f.readlines()
for line in lines[-10:]:
    print(line.rstrip())

# Check checkpoints
import glob
checkpoints = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")))
print(f"\n📁 CHECKPOINTS SAVED: {len(checkpoints)}")
for ck in checkpoints:
    size = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(ck)
        for f in files
    )
    print(f"   {os.path.basename(ck)}  →  {size/1024**2:.0f} MB")

# Check if training finished
if any("1000" in os.path.basename(c) for c in checkpoints):
    print("\n✅ TRAINING COMPLETE — checkpoint-1000 exists!")
    print("   Ready for inference")
elif len(checkpoints) > 0:
    last = os.path.basename(checkpoints[-1])
    step = last.split("-")[-1]
    print(f"\n⏳ STILL TRAINING — at step {step}/1000")
    print(f"   Wait for checkpoint-1000 before inference")
else:
    print("\n❌ No checkpoints yet — training still running")

📋 LAST 10 LINES OF TRAINING LOG:

Training completed. Do not forget to share your model on huggingface.co/models =)




{'train_runtime': 13237.8421, 'train_samples_per_second': 0.604, 'train_steps_per_second': 0.076, 'train_loss': 2.1334721426963807, 'epoch': 1.0}

100%|██████████| 1000/1000 [3:40:37<00:00, 13.24s/it]

📁 CHECKPOINTS SAVED: 3
   checkpoint-1000  →  507 MB
   checkpoint-600  →  507 MB
   checkpoint-800  →  507 MB

✅ TRAINING COMPLETE — checkpoint-1000 exists!
   Ready for inference


In [25]:
import os

CHECKPOINT = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-1000"

print("📦 Checkpoint contents:")
print("=" * 50)
for f in sorted(os.listdir(CHECKPOINT)):
    size = os.path.getsize(os.path.join(CHECKPOINT, f))
    print(f"  {f:<45} {size/1024**2:.1f} MB")

# Check adapter exists
adapter = os.path.join(CHECKPOINT, "adapter_model.safetensors")
if os.path.exists(adapter):
    size = os.path.getsize(adapter) / 1024**2
    print(f"\n✅ YOUR FINE-TUNED WEIGHTS CONFIRMED!")
    print(f"   adapter_model.safetensors = {size:.1f} MB")
    print(f"   This IS your trained model")
else:
    # check bin format
    adapter_bin = os.path.join(CHECKPOINT, "adapter_model.bin")
    if os.path.exists(adapter_bin):
        size = os.path.getsize(adapter_bin) / 1024**2
        print(f"\n✅ YOUR FINE-TUNED WEIGHTS CONFIRMED!")
        print(f"   adapter_model.bin = {size:.1f} MB")
    else:
        print("\n❌ Adapter file not found — list above to check filename")

📦 Checkpoint contents:
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                     187.9 MB
  optimizer.pt                                  319.1 MB
  rng_state.pth                                 0.0 MB
  scheduler.pt                                  0.0 MB
  trainer_state.json                            0.0 MB
  training_args.bin                             0.0 MB

✅ YOUR FINE-TUNED WEIGHTS CONFIRMED!
   adapter_model.safetensors = 187.9 MB
   This IS your trained model


In [26]:
import torch, sys, os, numpy as np
from PIL import Image

# ── Paths ──────────────────────────────────────────────────────
RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
CHECKPOINT   = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-1000"
NORMALIZER   = "/home/rishabh/Downloads/umi-pipeline-training/normalizer.pt"
DATASET_PATH = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
INSTRUCTION  = "pick up the marker and place it in the box"

sys.path.insert(0, RDT2_DIR)
os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION'] = 'eager'

# ── Load Models ────────────────────────────────────────────────
print("=" * 60)
print("Loading fine-tuned RDT2...")
print("=" * 60)

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from models.normalizer.normalizer import LinearNormalizer

print("[1/4] Loading base model...")
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "robotics-diffusion-transformer/RDT2-VQ",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    device_map="cuda"
)

print("[2/4] Loading YOUR fine-tuned adapter...")
model = PeftModel.from_pretrained(base, CHECKPOINT)
model.eval()
print(f"      Adapter loaded from {CHECKPOINT}")

print("[3/4] Loading processor...")
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

print("[4/4] Loading normalizer...")
normalizer = LinearNormalizer.load(NORMALIZER)

print("\n✅ All loaded!\n")

# ── Load Test Image from YOUR Dataset ─────────────────────────
print("Loading test image from your dataset...")
import zarr
root    = zarr.open(DATASET_PATH, mode='r')
img_arr = np.array(root['data']['camera0_rgb'][500], dtype=np.uint8)
img     = Image.fromarray(img_arr).resize((768, 384))
img.save("/tmp/test_frame.jpg")
print(f"✅ Image shape: {img_arr.shape} → resized 768x384")

# ── Run Inference ──────────────────────────────────────────────
print(f"\n🧠 Running inference...")
print(f"   Instruction: '{INSTRUCTION}'")

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": img},
        {"type": "text",  "text": INSTRUCTION}
    ]
}]

text   = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = processor(
    text=[text], images=[img], return_tensors="pt"
).to("cuda", torch.bfloat16)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
    )

# Decode output
input_len = inputs["input_ids"].shape[1]
generated = output_ids[:, input_len:]
decoded   = processor.batch_decode(
    generated, skip_special_tokens=True
)[0]

print(f"\n📤 RAW MODEL OUTPUT:")
print(f"   {decoded}")
print(f"\n   Output token IDs: {generated[0].tolist()[:10]}...")
print(f"\n✅ Inference working!")
print(f"   Next step: decode these tokens → action → send to M750")

Loading fine-tuned RDT2...


/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[1/4] Loading base model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.45it/s]


[2/4] Loading YOUR fine-tuned adapter...
      Adapter loaded from /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-1000
[3/4] Loading processor...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[4/4] Loading normalizer...
LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/normalizer.pt

✅ All loaded!

Loading test image from your dataset...
✅ Image shape: (224, 224, 3) → resized 768x384

🧠 Running inference...
   Instruction: 'pick up the marker and place it in the box'


/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(



📤 RAW MODEL OUTPUT:
   ᨁ긿⮞쑝ﭒ𝑼깄℁瑩𝙭Ἦ⧼爐켇𝚟뷗ﲀ샙狀쌰⚣℣ꪱ⠇𐤓Ⱀረ

   Output token IDs: [151650, 151534, 150863, 150738, 150991, 151161, 151308, 150864, 150626, 151140]...

✅ Inference working!
   Next step: decode these tokens → action → send to M750


In [33]:
# First find the correct VAE path in RDT2 repo
import os, subprocess

RDT2_DIR = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"

print("Finding VAE module in RDT2...")
for root, dirs, files in os.walk(RDT2_DIR):
    for f in files:
        if 'vae' in f.lower() and f.endswith('.py'):
            path = os.path.join(root, f)
            rel  = os.path.relpath(path, RDT2_DIR)
            print(f"  Found: {rel}")

print("\nFinding MultiVQVAE class...")
result = subprocess.run(
    ['grep', '-r', 'MultiVQVAE', RDT2_DIR, '--include=*.py', '-l'],
    capture_output=True, text=True
)
print(result.stdout)

print("\nFinding class definitions with 'decode'...")
result2 = subprocess.run(
    ['grep', '-r', 'class.*VAE\|class.*Tokenizer\|class.*Action', 
     RDT2_DIR, '--include=*.py'],
    capture_output=True, text=True
)
print(result2.stdout[:2000])

Finding VAE module in RDT2...
  Found: vqvae/models/vqvae.py
  Found: vqvae/models/multivqvae.py

Finding MultiVQVAE class...
/home/rishabh/Downloads/umi-pipeline-training/RDT2/vqvae/models/multivqvae.py
/home/rishabh/Downloads/umi-pipeline-training/RDT2/deploy/inference_real_vq.py
/home/rishabh/Downloads/umi-pipeline-training/RDT2/train.py


Finding class definitions with 'decode'...
/home/rishabh/Downloads/umi-pipeline-training/RDT2/vqvae/models/vqvae.py:class VQVAE(nn.Module):
/home/rishabh/Downloads/umi-pipeline-training/RDT2/vqvae/models/multivqvae.py:class MultiVQVAE(nn.Module, PyTorchModelHubMixin):
/home/rishabh/Downloads/umi-pipeline-training/RDT2/deploy/umi/common/timestamp_accumulator.py:class TimestampActionAccumulator:



<>:23: SyntaxWarning: invalid escape sequence '\|'
<>:23: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_4096/663195556.py:23: SyntaxWarning: invalid escape sequence '\|'
  ['grep', '-r', 'class.*VAE\|class.*Tokenizer\|class.*Action',


In [35]:
import torch, sys, os, numpy as np
from PIL import Image

RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
CHECKPOINT   = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-1000"
NORMALIZER   = "/home/rishabh/Downloads/umi-pipeline-training/normalizer.pt"
DATASET_PATH = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
INSTRUCTION  = "pick up the marker and place it in the box"

# Add ALL paths
sys.path.insert(0, RDT2_DIR)
sys.path.insert(0, os.path.join(RDT2_DIR, 'vqvae'))

os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION'] = 'eager'
os.environ['PYTORCH_CUDA_ALLOC_CONF']          = 'expandable_segments:True'
os.environ['CUDA_LAUNCH_BLOCKING']             = '1'

print("✅ Paths set")
print(f"   RDT2:       {RDT2_DIR}")
print(f"   Checkpoint: {CHECKPOINT}")

✅ Paths set
   RDT2:       /home/rishabh/Downloads/umi-pipeline-training/RDT2
   Checkpoint: /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-1000


In [36]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from models.normalizer.normalizer import LinearNormalizer
from models.multivqvae import MultiVQVAE   # correct path!

print("[1/4] Loading base model...")
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "robotics-diffusion-transformer/RDT2-VQ",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    device_map="cuda"
)

print("[2/4] Loading your fine-tuned adapter...")
model = PeftModel.from_pretrained(base, CHECKPOINT)
model.eval()

print("[3/4] Loading processor...")
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", use_fast=True
)

print("[4/4] Loading VAE + normalizer...")
vae = MultiVQVAE.from_pretrained(
    "robotics-diffusion-transformer/RVQActionTokenizer"
).cuda().eval()

normalizer = LinearNormalizer.load(NORMALIZER)

print("\n✅ ALL MODELS LOADED!")

[1/4] Loading base model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
